# Task 3 — демо-сессия для двух ролей экспертов в Google Colab

Этот блокнот показывает **две роли** в A/B процессе для Task 3 в `top-papers-graph`:

1. **Эксперт-создатель теста** — готовит test set, trajectory bundle, creator manifest и offline-форму.
2. **Эксперт-участник теста** — получает blind A/B пакет, открывает offline HTML и сдаёт JSON/CSV с оценками.

Блокнот сделан как **демо-сессия**: он использует реальные файлы и реальные примеры из репозитория, который загружается через `git clone`, и при этом не требует тяжёлого прогона моделей ради демонстрации процесса.


## Что получится в конце

После прохождения обеих частей у вас будут:

- `creator_demo/task1_demo_bundle.zip` — минимальный trajectory bundle;
- `creator_demo/task3_ab_creator_demo_form.html` — offline-форма для эксперта-создателя;
- `creator_demo/task3_ab_case_manifest.demo.json` — демо-manifest тестового набора;
- `participant_demo/expert_*_bundle.zip` — blind A/B пакет для эксперта-участника;
- `participant_demo/task3_review_filled_demo.json` — пример итоговой экспертной разметки.


In [ ]:
# @title 0. Установка и клонирование репозитория
!pip -q install pyyaml nbformat ipywidgets

import json
import sys
import html
import zipfile
import shutil
from copy import deepcopy
from pathlib import Path

import yaml
from IPython.display import display, Markdown

REPO_URL = 'https://github.com/top-papers/top-papers-graph.git'
REPO_DIR = Path('/content/top-papers-graph')
WORK_DIR = Path('/content/task3_expert_demo')
WORK_DIR.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    !git clone --depth 1 {REPO_URL} {REPO_DIR}

if str(REPO_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_DIR / 'src'))

print('REPO_DIR =', REPO_DIR)
print('WORK_DIR =', WORK_DIR)


In [ ]:
# @title 1. Быстрый аудит репозитория: какими файлами мы пользуемся в демо
KEY_FILES = [
    'task1_reasoning_trajectories_onine_offline_forms.ipynb',
    'task3_dual_local_models_blind_ab.ipynb',
    'tmp_smoke_demo.yaml',
    'data/experts/trajectories/_template.yaml',
    'tests/test_task3_pipeline.py',
    'tests/test_task3_dual_model_review.py',
    'src/scireason/task3_dual_model_review.py',
    'data/experts/mm_ab_reviews/task3_mm_ab_review_template.json',
    'data/experts/mm_ab_reviews/task3_ab_case_manifest.reviewer_fatigue_optimized.18.json',
]
for rel in KEY_FILES:
    path = REPO_DIR / rel
    print(('OK   ' if path.exists() else 'MISS '), rel)


In [ ]:
# @title 2. Вспомогательные функции

def show_text(path, max_chars=2400):
    path = Path(path)
    print(f'\n===== {path} =====')
    text = path.read_text(encoding='utf-8')
    print(text[:max_chars])
    if len(text) > max_chars:
        print('\n... [truncated] ...')


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    return path


def write_yaml(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(yaml.safe_dump(payload, allow_unicode=True, sort_keys=False), encoding='utf-8')
    return path


def zip_paths(output_zip, items):
    output_zip = Path(output_zip)
    output_zip.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(output_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for src, arcname in items:
            src = Path(src)
            if src.exists() and src.is_file():
                zf.write(src, arcname=arcname)
    return output_zip


# Часть 1. Эксперт-создатель тестового набора

## Для чего нужна эта работа

В Task 3 эксперт-создатель нужен не для того, чтобы “просто запустить модели”, а для того, чтобы **сконструировать правильный тест**. Именно он задаёт, где blind A/B должен быть чувствителен к реальной разнице между baseline VLM и tuned VLM.

Главная цель этой роли:

- выбрать правильные случаи сравнения;
- зафиксировать, **на что должен смотреть эксперт-участник**;
- сделать тест чувствительным к ошибкам типа `missed_visual_fact`, `wrong_evidence_linkage`, `needs_time_fix`, `hallucinated_visual_inference`.

Простыми словами: **создатель теста проектирует сам экзамен** для Task 3.


## Шаг 1. Смотрим реальные файлы репозитория

Мы опираемся на реальные исходники:

- `data/experts/trajectories/_template.yaml` — минимальный trajectory template;
- `tmp_smoke_demo.yaml` — маленький живой пример YAML;
- `task1_reasoning_trajectories_onine_offline_forms.ipynb` — форма Task 1;
- `tests/test_task3_pipeline.py` — пример synthetic `processed_papers`;
- `data/experts/mm_ab_reviews/*.json` — примеры manifest и review templates, если они есть в вашей версии repo.


In [ ]:
# @title 3. Просмотр примеров для роли создателя
for path in [
    REPO_DIR / 'data/experts/trajectories/_template.yaml',
    REPO_DIR / 'tmp_smoke_demo.yaml',
]:
    if path.exists():
        show_text(path, max_chars=1800)

sample_manifest = REPO_DIR / 'data/experts/mm_ab_reviews/task3_ab_case_manifest.reviewer_fatigue_optimized.18.json'
if sample_manifest.exists():
    show_text(sample_manifest, max_chars=2200)
else:
    print('Пример case manifest не найден — notebook создаст демо-версию сам.')


## Шаг 2. Собираем минимальный trajectory bundle для демо

В реальной работе эксперт-создатель может сначала заполнить полноценную форму Task 1. Для демо мы показываем короткий путь:

- берём за основу `tmp_smoke_demo.yaml` или `_template.yaml`;
- подставляем демо-topic;
- добавляем paper identifiers;
- сохраняем готовый YAML и ZIP.


In [ ]:
# @title 4. Создание демо trajectory YAML и ZIP
CREATOR_DIR = WORK_DIR / 'creator_demo'
CREATOR_DIR.mkdir(parents=True, exist_ok=True)

base_yaml_path = REPO_DIR / 'tmp_smoke_demo.yaml'
if base_yaml_path.exists():
    trajectory_payload = yaml.safe_load(base_yaml_path.read_text(encoding='utf-8'))
else:
    trajectory_payload = {'submission_id': 'demo_creator_submission', 'topic': 'temporal catalyst latency forecasting', 'papers': [], 'steps': []}

trajectory_payload['submission_id'] = 'demo_creator_submission'
trajectory_payload['topic'] = 'temporal catalyst latency forecasting'
trajectory_payload['papers'] = [
    {'id': 'doi:10.1234/demo.paper.1', 'title': 'Temporal catalyst study', 'year': 2023},
    {'id': 'doi:10.1234/demo.paper.2', 'title': 'Temporal catalyst follow-up', 'year': 2024},
]
trajectory_payload['steps'] = [
    {
        'id': 'step_1',
        'inference': 'Сначала проверяем, где ключевой факт лежит в тексте, а где в figure/table.',
        'next_question': 'Какие кейсы будут multimodal_hard?',
        'sources': [
            {'source': 'doi:10.1234/demo.paper.1', 'title': 'Temporal catalyst study'},
            {'source': 'doi:10.1234/demo.paper.2', 'title': 'Temporal catalyst follow-up'},
        ],
    }
]

trajectory_yaml = write_yaml(CREATOR_DIR / 'task1_demo_trajectory.yaml', trajectory_payload)
trajectory_zip = zip_paths(CREATOR_DIR / 'task1_demo_bundle.zip', [(trajectory_yaml, 'task1_demo_trajectory.yaml')])
print('Создан YAML:', trajectory_yaml)
print('Создан ZIP:', trajectory_zip)
show_text(trajectory_yaml, max_chars=2200)


## Шаг 3. Делаем маленький `processed_papers` для демо

Ниже используется реальный паттерн из `tests/test_task3_pipeline.py`: создаются paper directories, `meta.json`, `chunks.jsonl` и `mm/pages.jsonl`.

Это не продовый корпус, а **демо-данные**, но они повторяют контракт файлов Task 3.


In [ ]:
# @title 5. Сборка demo processed_papers
PROCESSED_DIR = CREATOR_DIR / 'processed_papers_demo'
if PROCESSED_DIR.exists():
    shutil.rmtree(PROCESSED_DIR)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

def write_processed_paper(root: Path, *, paper_id: str, year: int, title: str, text: str, mm_text: str, table_md: str, caption: str):
    paper_dir = root / paper_id
    (paper_dir / 'mm').mkdir(parents=True, exist_ok=True)
    write_json(paper_dir / 'meta.json', {'id': paper_id, 'title': title, 'year': year})
    (paper_dir / 'chunks.jsonl').write_text(
        json.dumps({'chunk_id': f'{paper_id}:c1', 'paper_id': paper_id, 'text': text, 'modality': 'text', 'source_backend': 'demo_notebook'}, ensure_ascii=False) + '\n',
        encoding='utf-8',
    )
    (paper_dir / 'mm' / 'pages.jsonl').write_text(
        json.dumps({'paper_id': paper_id, 'page': 1, 'text': mm_text, 'image_path': '', 'vlm_caption': caption, 'tables_md': table_md, 'equations_md': ''}, ensure_ascii=False) + '\n',
        encoding='utf-8',
    )

write_processed_paper(PROCESSED_DIR, paper_id='paper-1', year=2023, title='Temporal catalyst study', text='Catalyst A reduces latency in 2023 trials. Forecast accuracy improves after catalyst A deployment.', mm_text='Page evidence: catalyst A, latency reduction, 2023 timeline.', table_md='Year | effect\n2023 | latency reduction', caption='Figure caption: catalyst A lowers latency in 2023 trials.')
write_processed_paper(PROCESSED_DIR, paper_id='paper-2', year=2024, title='Temporal catalyst follow-up', text='Catalyst A reduces latency and improves forecast stability in 2024 monitoring.', mm_text='Figure summary: catalyst A maintains lower latency in 2024.', table_md='Year | effect\n2024 | latency reduction', caption='Figure caption: catalyst A maintains lower latency in 2024 monitoring.')

print('Demo processed dir:', PROCESSED_DIR)
for p in sorted(PROCESSED_DIR.rglob('*')):
    if p.is_file():
        print(' -', p.relative_to(PROCESSED_DIR))


## Шаг 4. Создаём creator manifest и offline-форму

Теперь главный шаг роли создателя: подготовить **список кейсов**, по которым потом будет строиться blind A/B.

В демо ниже мы делаем так:

- если в репозитории уже есть пример `task3_ab_case_manifest.reviewer_fatigue_optimized.18.json`, берём его как реальный пример;
- иначе собираем маленький manifest прямо в notebook;
- создаём **offline HTML форму**, в которой создатель теста может редактировать кейсы и экспортировать заполненный JSON.


In [ ]:
# @title 6. Создание demo case manifest и creator offline HTML
sample_manifest_path = REPO_DIR / 'data/experts/mm_ab_reviews/task3_ab_case_manifest.reviewer_fatigue_optimized.18.json'

if sample_manifest_path.exists():
    case_manifest = json.loads(sample_manifest_path.read_text(encoding='utf-8'))
    case_manifest = deepcopy(case_manifest)
    case_manifest['experiment_meta']['submission_id'] = 'demo_creator_submission'
    case_manifest['experiment_meta']['creator_id'] = 'demo_creator'
    case_manifest['cases'] = case_manifest['cases'][:6]
else:
    case_manifest = {
        'schema_version': 'task3_ab_case_manifest_v1',
        'generated_at': '2026-04-20T00:00:00Z',
        'experiment_meta': {
            'topic': 'temporal catalyst latency forecasting',
            'submission_id': 'demo_creator_submission',
            'creator_id': 'demo_creator',
            'cutoff_year': '2024',
            'review_goal': 'Demo creator session for Task 3 case-based blind A/B',
        },
        'strata_targets': {'multimodal_hard': 3, 'temporal_hard': 2, 'easy_control': 1},
        'error_taxonomy': ['missed_visual_fact', 'wrong_evidence_linkage', 'needs_time_fix', 'hallucinated_visual_inference'],
        'cases': [
            {
                'case_id': 'CASE-001', 'enabled': True, 'primary_endpoint': True, 'stratum': 'multimodal_hard',
                'paper_title': 'Temporal catalyst follow-up', 'paper_id': 'paper-2', 'year': '2024', 'evidence_kind': 'figure',
                'page_hint': 'p.1 main figure: lower latency in 2024',
                'creator_prompt': 'Проверь, извлечён ли visual fact про lower latency из figure.',
                'creator_rationale': 'Кейс на missed_visual_fact.',
                'review_focus': ['evidence', 'visual_fact'],
                'expected_error_modes': ['missed_visual_fact'],
                'match': {'candidate_target_contains': 'latency', 'rank_hint': '1'},
                'notes': 'demo',
            }
        ],
    }

manifest_path = write_json(CREATOR_DIR / 'task3_ab_case_manifest.demo.json', case_manifest)

def build_creator_html_form(manifest_payload: dict, output_html: Path):
    app_json = json.dumps(manifest_payload, ensure_ascii=False).replace('</', '<\/')
    template = """
<!doctype html>
<html lang='ru'>
<head>
  <meta charset='utf-8'>
  <title>Task 3 creator form</title>
  <style>
    body { font-family: Arial, sans-serif; margin: 24px; line-height: 1.35; }
    .case { border: 1px solid #ddd; border-radius: 12px; padding: 14px; margin: 14px 0; }
    .row { display: grid; grid-template-columns: 180px 1fr; gap: 8px; margin: 6px 0; }
    input, textarea { width: 100%; box-sizing: border-box; padding: 8px; }
    textarea { min-height: 72px; }
    .toolbar { position: sticky; top: 0; background: #fff; padding: 10px 0; border-bottom: 1px solid #eee; }
    button { padding: 10px 14px; margin-right: 8px; }
    code { background: #f6f6f6; padding: 2px 4px; }
  </style>
</head>
<body>
  <div class='toolbar'>
    <button onclick='downloadJson()'>Скачать filled JSON</button>
    <button onclick='saveLocal()'>Сохранить в браузере</button>
    <button onclick='loadLocal()'>Восстановить из браузера</button>
  </div>
  <h1>Task 3 — offline creator form</h1>
  <p>Поправьте кейсы и сохраните <code>task3_ab_case_manifest.filled.json</code>.</p>
  <div id='root'></div>
  <script>
    const appData = __APP_DATA__;
    function val(x) { return x === undefined || x === null ? '' : x; }
    function render() {
      const root = document.getElementById('root');
      root.innerHTML = '';
      appData.cases.forEach((item, idx) => {
        const wrap = document.createElement('div');
        wrap.className = 'case';
        wrap.innerHTML = `
          <h3>${item.case_id} — ${item.stratum}</h3>
          <div class='row'><label>paper_title</label><input data-k='paper_title' data-i='${idx}' value='${val(item.paper_title)}'></div>
          <div class='row'><label>paper_id</label><input data-k='paper_id' data-i='${idx}' value='${val(item.paper_id)}'></div>
          <div class='row'><label>evidence_kind</label><input data-k='evidence_kind' data-i='${idx}' value='${val(item.evidence_kind)}'></div>
          <div class='row'><label>page_hint</label><textarea data-k='page_hint' data-i='${idx}'>${val(item.page_hint)}</textarea></div>
          <div class='row'><label>creator_prompt</label><textarea data-k='creator_prompt' data-i='${idx}'>${val(item.creator_prompt)}</textarea></div>
          <div class='row'><label>creator_rationale</label><textarea data-k='creator_rationale' data-i='${idx}'>${val(item.creator_rationale)}</textarea></div>
          <div class='row'><label>match.rank_hint</label><input data-k='match.rank_hint' data-i='${idx}' value='${val((item.match||{}).rank_hint)}'></div>
          <div class='row'><label>match.candidate_target_contains</label><input data-k='match.candidate_target_contains' data-i='${idx}' value='${val((item.match||{}).candidate_target_contains)}'></div>
          <div class='row'><label>notes</label><textarea data-k='notes' data-i='${idx}'>${val(item.notes)}</textarea></div>
        `;
        root.appendChild(wrap);
      });
      root.querySelectorAll('input,textarea').forEach(el => {
        el.addEventListener('input', (e) => {
          const i = Number(e.target.dataset.i);
          const k = e.target.dataset.k;
          if (k.startsWith('match.')) {
            appData.cases[i].match = appData.cases[i].match || {};
            appData.cases[i].match[k.slice(6)] = e.target.value;
          } else {
            appData.cases[i][k] = e.target.value;
          }
        });
      });
    }
    function downloadJson() {
      const blob = new Blob([JSON.stringify(appData, null, 2)], {type: 'application/json'});
      const a = document.createElement('a');
      a.href = URL.createObjectURL(blob);
      a.download = 'task3_ab_case_manifest.filled.json';
      a.click();
    }
    function saveLocal() { localStorage.setItem('task3_creator_form', JSON.stringify(appData)); alert('Сохранено'); }
    function loadLocal() {
      const raw = localStorage.getItem('task3_creator_form');
      if (!raw) return alert('В localStorage ничего нет');
      const parsed = JSON.parse(raw);
      Object.assign(appData, parsed);
      render();
      alert('Восстановлено');
    }
    render();
  </script>
</body>
</html>
"""
    output_html.write_text(template.replace('__APP_DATA__', app_json), encoding='utf-8')
    return output_html

creator_html = build_creator_html_form(case_manifest, CREATOR_DIR / 'task3_ab_creator_demo_form.html')
creator_bundle_zip = zip_paths(
    CREATOR_DIR / 'task3_ab_creator_bundle.zip',
    [
        (manifest_path, 'task3_ab_case_manifest.demo.json'),
        (creator_html, 'task3_ab_creator_demo_form.html'),
        (trajectory_yaml, 'task1_demo_trajectory.yaml'),
    ],
)

print('Manifest:', manifest_path)
print('Offline creator form:', creator_html)
print('Creator bundle zip:', creator_bundle_zip)


## Что должен сделать создатель теста после этой части

Итог роли создателя:

1. проверить и поправить кейсы в `task3_ab_creator_demo_form.html`;
2. сохранить `task3_ab_case_manifest.filled.json`;
3. передать дальше три вещи:
   - trajectory bundle,
   - filled case manifest,
   - curated `processed_papers`.


# Часть 2. Эксперт-участник blind A/B теста

## Для чего нужна эта работа

Эксперт-участник не проектирует тест, а **судит результаты**. Его задача — в blind-режиме сравнить две анонимные системы и сказать, какая лучше:

- извлекла evidence;
- удержала temporal scope;
- не потеряла visual fact;
- не придумала лишнюю визуальную интерпретацию.

В контексте всего проекта это превращает сырые A/B outputs в **валидное human evaluation**, на которое потом можно опираться при сравнении baseline VLM и SFT/DPO VLM в Task 3.


## Шаг 1. Смотрим реальные файлы для blind review

Нас интересуют:

- `src/scireason/task3_dual_model_review.py` — builder слепого A/B пакета;
- `tests/test_task3_dual_model_review.py` — минимальный живой пример двух вариантов output;
- `task3_dual_local_models_blind_ab.ipynb` — notebook-оркестратор;
- `data/experts/mm_ab_reviews/task3_mm_ab_review_template.json` — шаблон ответа эксперта, если он есть в вашей версии repo.


In [ ]:
# @title 7. Просмотр примеров для роли участника
for path in [
    REPO_DIR / 'src/scireason/task3_dual_model_review.py',
    REPO_DIR / 'tests/test_task3_dual_model_review.py',
]:
    if path.exists():
        show_text(path, max_chars=2200)

review_template_path = REPO_DIR / 'data/experts/mm_ab_reviews/task3_mm_ab_review_template.json'
if review_template_path.exists():
    show_text(review_template_path, max_chars=1500)


## Шаг 2. Генерируем минимальный blind A/B пакет на демо-данных

Чтобы не ждать настоящий длинный прогон моделей, мы делаем **демо-сборку** blind review пакета на маленьких synthetic outputs.

Это тот же формат артефактов, который использует код репозитория и его тесты:

- `hypotheses_ranked.json`
- `hypotheses_ranked.md`
- `task3_manifest.json`
- offline HTML для эксперта
- ZIP пакет без owner-only key

Если в вашей версии repo есть case-based builder, notebook попробует использовать его. Если нет — будет использован стандартный dual-model blind review builder.


In [ ]:
# @title 8. Построение demo blind review bundle
PARTICIPANT_DIR = WORK_DIR / 'participant_demo'
if PARTICIPANT_DIR.exists():
    shutil.rmtree(PARTICIPANT_DIR)
PARTICIPANT_DIR.mkdir(parents=True, exist_ok=True)

from scireason.task3_dual_model_review import build_task3_dual_model_expert_bundle
try:
    from scireason.task3_ab_case_manifest import build_task3_case_based_expert_bundle
    HAS_CASE_BASED = True
except Exception:
    HAS_CASE_BASED = False

def write_demo_bundle(bundle_dir: Path, hypotheses: list, runtime_label: str):
    bundle_dir.mkdir(parents=True, exist_ok=True)
    write_json(bundle_dir / 'hypotheses_ranked.json', hypotheses)
    (bundle_dir / 'hypotheses_ranked.md').write_text('# demo hypotheses\n', encoding='utf-8')
    manifest = {
        'bundle_dir': str(bundle_dir),
        'query': 'temporal catalyst latency forecasting',
        'runtime': {'vlm_backend': 'demo_local', 'vlm_model_id': runtime_label},
        'artifacts': {
            'hypotheses_ranked': str(bundle_dir / 'hypotheses_ranked.json'),
            'hypotheses_markdown': str(bundle_dir / 'hypotheses_ranked.md'),
        },
    }
    manifest_path = bundle_dir / 'task3_manifest.json'
    write_json(manifest_path, manifest)
    return manifest_path

hypo_a = [{
    'rank': 1,
    'candidate': {'kind': 'missing_link', 'source': 'Catalyst A', 'predicate': 'improves', 'target': 'forecast stability', 'score': 0.82, 'time_scope': '2023-2024'},
    'temporal_context': {'ordering': 'strengthening', 'years': [2023, 2024], 'time_scope': '2023-2024'},
    'hypothesis': {
        'title': 'Catalyst A may improve forecast stability over time',
        'premise': 'Figure and table both suggest lower latency and more stable forecasting in 2024.',
        'mechanism': 'Catalyst A may progressively reduce variance after deployment.',
        'time_scope': '2023-2024',
        'proposed_experiment': 'Replicate on held-out monitoring data.',
        'supporting_evidence': [
            {'source_id': 'paper-2', 'text_snippet': 'Figure shows lower latency in 2024.', 'page': 1},
            {'source_id': 'paper-2', 'text_snippet': 'Table row confirms stability benefit in 2024.', 'page': 1},
        ],
    },
    'final_score': 0.91,
}]

hypo_b = [{
    'rank': 1,
    'candidate': {'kind': 'missing_link', 'source': 'Catalyst A', 'predicate': 'improves', 'target': 'forecast stability', 'score': 0.76, 'time_scope': '2023-2024'},
    'temporal_context': {'ordering': 'predicted_or_missing', 'years': [2023, 2024], 'time_scope': '2023-2024'},
    'hypothesis': {
        'title': 'Catalyst A may be associated with later stability',
        'premise': 'Signals suggest a possible relation in later observations.',
        'mechanism': 'Potential delayed stabilization after intervention.',
        'time_scope': '2023-2024',
        'proposed_experiment': 'Run a later-window replication study.',
        'supporting_evidence': [
            {'source_id': 'paper-2', 'text_snippet': 'Later-stage measurements remain more stable.', 'page': 1},
        ],
    },
    'final_score': 0.86,
}]

manifest_a = write_demo_bundle(PARTICIPANT_DIR / 'variant_alpha', hypo_a, runtime_label='/models/base_demo')
manifest_b = write_demo_bundle(PARTICIPANT_DIR / 'variant_beta', hypo_b, runtime_label='/models/tuned_demo')
task_meta = {'topic': 'temporal catalyst latency forecasting', 'submission_id': 'demo_creator_submission'}
case_manifest_for_participant = CREATOR_DIR / 'task3_ab_case_manifest.demo.json'

if HAS_CASE_BASED and case_manifest_for_participant.exists():
    expert_bundle_zip = build_task3_case_based_expert_bundle(
        manifest_a, manifest_b, case_manifest_for_participant, task_meta=task_meta,
        output_path=PARTICIPANT_DIR / 'expert_case_based_blind_review_bundle.zip',
        model_a_descriptor={'vlm_model_id': '/models/base_demo'},
        model_b_descriptor={'vlm_model_id': '/models/tuned_demo'},
    )
else:
    expert_bundle_zip = build_task3_dual_model_expert_bundle(
        manifest_a, manifest_b, task_meta=task_meta,
        output_path=PARTICIPANT_DIR / 'expert_dual_model_blind_review_bundle.zip',
        model_a_descriptor={'vlm_model_id': '/models/base_demo'},
        model_b_descriptor={'vlm_model_id': '/models/tuned_demo'},
    )

print('Expert bundle:', expert_bundle_zip)
for p in sorted(PARTICIPANT_DIR.rglob('*')):
    if p.is_file():
        print(' -', p.relative_to(PARTICIPANT_DIR))


## Шаг 3. Подготавливаем шаблон ответа участника

Участник blind review должен вернуть JSON с verdicts по парам. Ниже notebook автоматически строит такой шаблон по найденному public manifest.


In [ ]:
# @title 9. Создание шаблона и примера заполненного review JSON
public_manifest_candidates = list(PARTICIPANT_DIR.rglob('*blind_review_manifest.json')) + list(PARTICIPANT_DIR.rglob('*review_manifest.json'))
public_manifest_path = public_manifest_candidates[0] if public_manifest_candidates else None
if public_manifest_path is None:
    raise FileNotFoundError('Не найден public manifest blind review пакета')

public_manifest = json.loads(public_manifest_path.read_text(encoding='utf-8'))
records = public_manifest.get('records') or []

review_template_repo = REPO_DIR / 'data/experts/mm_ab_reviews/task3_mm_ab_review_template.json'
if review_template_repo.exists():
    review_payload = json.loads(review_template_repo.read_text(encoding='utf-8'))
else:
    review_payload = {'artifact_version': 1, 'review_mode': 'task3_blind_ab', 'reviewer_id': '', 'submission_id': '', 'items': []}

review_payload['reviewer_id'] = 'demo_participant'
review_payload['submission_id'] = public_manifest.get('meta', {}).get('submission_id', 'demo_creator_submission')
review_payload['items'] = []
for rec in records:
    review_payload['items'].append({
        'pair_id': rec.get('pair_id', ''),
        'unit_id': rec.get('case_meta', {}).get('paper_id', '') if isinstance(rec.get('case_meta'), dict) else '',
        'preferred_variant': 'skip',
        'better_evidence': 'skip',
        'better_temporal': 'skip',
        'better_testability': 'skip',
        'better_novelty': 'skip',
        'global_verdict': 'skip',
        'error_tags': [],
        'confidence': 3,
        'comments': '',
    })

review_template_path = write_json(PARTICIPANT_DIR / 'task3_review_template.json', review_payload)
example_filled = deepcopy(review_payload)
if example_filled['items']:
    example_filled['items'][0].update({
        'preferred_variant': 'A',
        'better_evidence': 'A',
        'better_temporal': 'A',
        'global_verdict': 'accept',
        'error_tags': ['missed_visual_fact'],
        'confidence': 4,
        'comments': 'Вариант A лучше удерживает figure/table evidence и явнее сохраняет temporal scope.',
    })
filled_path = write_json(PARTICIPANT_DIR / 'task3_review_filled_demo.json', example_filled)
print('Public manifest:', public_manifest_path)
print('Review template:', review_template_path)
print('Filled example:', filled_path)
show_text(filled_path, max_chars=2200)


In [ ]:
# @title 10. Финальный список артефактов демо-сессии
print('\n=== CREATOR OUTPUTS ===')
for p in sorted(CREATOR_DIR.rglob('*')):
    if p.is_file():
        print(' -', p.relative_to(CREATOR_DIR))

print('\n=== PARTICIPANT OUTPUTS ===')
for p in sorted(PARTICIPANT_DIR.rglob('*')):
    if p.is_file():
        print(' -', p.relative_to(PARTICIPANT_DIR))


## Короткое резюме

### Что делает создатель теста
- определяет cases;
- собирает trajectory + curated processed papers;
- делает creator manifest и offline form.

### Что делает участник теста
- получает blind bundle;
- оценивает A/B пары по evidence / temporal / overall;
- сдаёт JSON с verdicts.

### Почему это полезно для Task 3
Такой процесс отделяет:
- **конструирование теста**,
- **генерацию blind артефактов**,
- **независимую экспертную оценку**.

Именно так проще увидеть реальные различия между baseline VLM и SFT/DPO VLM на hard multimodal и temporal cases.
